# 🥇 Proyecto Final: Análisis End-to-End Multidominio
Este notebook contiene el procesamiento, limpieza, análisis de sentimientos y visualización para **TODOS** los datasets del proyecto (~90 archivos, >500 MB).

## Contenido
1. **Configuración y Carga de Librerías**
2. **Fase 1: ETL y Limpieza Avanzada (Todos los Datasets)**
3. **Fase 2: Procesamiento Analítico y Análisis de Sentimientos (NLP)**
4. **Fase 3: Métricas Gráficas y Visualización**



In [2]:
# 1. Configuración y Carga de Librerías
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import re
import json

# Para NLP / Sentimientos
try:
    from textblob import TextBlob
except ImportError:
    !pip install textblob
    from textblob import TextBlob

# Configuración visual
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)

DIR_BASE = "../datasets"
print("Librerías cargadas correctamente.")



Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 625.0/625.0 kB 6.9 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 16.2 MB/s  0:00:00
   ---------------------------------------- 0.0/676.7 kB ? eta -:--:--
   ---------------------------------------- 676.7/676.7 kB 9.4 MB/s  0:00:00

   ---------------------------------------- 0/7 [tqdm]
   ---------------------------------------- 0/7 [tqdm]
   ---------------------------------------- 0/7 [tqdm]
   ---------------------------------------- 0/7 [tqdm]
   ---------------------------------------- 0/7 [tqdm]
   ---------------------------------------- 0/7 [tqdm]
   ---------------------------------------- 0/7 [tqdm]
   ----- ---------------------------------- 1/7 [regex]
   ----- --------------------------------

---
## Fase 1: ETL y Limpieza Avanzada
En esta sección cargaremos absolutamente todos los datasets de las distintas fuentes, aplicando limpiezas específicas según la naturaleza de los datos.



In [ ]:
# 1.1 Datos Deportivos Históricos (1896 - 2022)
print("Cargando datos históricos...")
df_athletes_hist = pd.read_csv(f"{DIR_BASE}/summer and winter 1896-2022/olympic_athletes.csv")
df_medals_hist = pd.read_csv(f"{DIR_BASE}/summer and winter 1896-2022/olympic_medals.csv")

# Limpieza: Rellenar nulos
df_athletes_hist['athlete_medals'] = df_athletes_hist['athlete_medals'].fillna('None')

print(f"Atletas Históricos: {df_athletes_hist.shape[0]} registros")
print(f"Medallas Históricas: {df_medals_hist.shape[0]} registros")



In [ ]:
# 1.2 Datos de París 2024
print("Cargando datos de París 2024...")
df_paris_athletes = pd.read_csv(f"{DIR_BASE}/paris 2024/athletes.csv")
df_paris_medals = pd.read_csv(f"{DIR_BASE}/paris 2024/medals.csv")

# Consolidando todos los resultados de París 2024 (leyendo 45 archivos)
archivos_resultados = glob.glob(f"{DIR_BASE}/paris 2024/results/*.csv")
lista_resultados = []
for archivo in archivos_resultados:
    try:
        temp_df = pd.read_csv(archivo)
        temp_df['Deporte'] = os.path.basename(archivo).replace(".csv", "")
        lista_resultados.append(temp_df)
    except:
        pass
df_paris_resultados = pd.concat(lista_resultados, ignore_index=True)

print(f"Atletas París: {df_paris_athletes.shape[0]}")
print(f"Resultados Consolidados París: {df_paris_resultados.shape[0]}")



In [ ]:
# 1.3 Datos de Tokio 2020
print("Cargando datos de Tokio 2020...")
df_tokyo_athletes = pd.read_excel(f"{DIR_BASE}/tokyo 2020/Athletes.xlsx")
df_tokyo_medals = pd.read_excel(f"{DIR_BASE}/tokyo 2020/Medals.xlsx")
print(f"Atletas Tokio: {df_tokyo_athletes.shape[0]}")



In [ ]:
# 1.4 Contexto Socioeconómico (PIB, Población, IDH)
print("Cargando datos socioeconómicos...")
# IDH (Melt para normalizar >1100 columnas)
df_idh = pd.read_csv(f"{DIR_BASE}/indiceDeDesarrolloHumano/HDR25_Composite_indices_complete_time_series.csv")
id_vars = list(df_idh.columns[:5])
df_idh_melt = df_idh.melt(id_vars=id_vars, var_name='Indicador_Anio', value_name='Valor')
# Extraer el año y la métrica
df_idh_melt['Anio'] = df_idh_melt['Indicador_Anio'].str.extract(r'(_[0-9]{4})$')[0].str.replace('_', '')
df_idh_melt['Metrica'] = df_idh_melt['Indicador_Anio'].str.replace(r'(_[0-9]{4})$', '', regex=True)

# PIB y Población (JSON)
with open(f"{DIR_BASE}/PIByDatosPoblacionales/api_rest_pib_historico.json", "r", encoding="utf-8") as f:
    df_pib = pd.DataFrame(json.load(f))
    
with open(f"{DIR_BASE}/PIByDatosPoblacionales/api_rest_poblacion_paises.json", "r", encoding="utf-8") as f:
    df_pob = pd.DataFrame(json.load(f))

print(f"IDH Normalizado (Melt): {df_idh_melt.shape[0]} registros")
print(f"PIB: {df_pib.shape[0]}, Población: {df_pob.shape[0]}")



In [ ]:
# 1.5 Clima Histórico Distribuido
print("Cargando clima (CSV, JSON, SQLite)...")
df_clima_america = pd.read_csv(f"{DIR_BASE}/clima_historico_distribuido/clima_america_nosql.csv")

# SQLite
conn = sqlite3.connect(f"{DIR_BASE}/clima_historico_distribuido/clima_mysql.db")
try:
    df_clima_sql = pd.read_sql("SELECT * FROM clima", conn)
except:
    # Obtener nombre de tabla dinamicamente si 'clima' no existe
    tbl = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn).iloc[0,0]
    df_clima_sql = pd.read_sql(f"SELECT * FROM {tbl}", conn)
conn.close()

# Europa JSON (combinando partes)
lista_europa = []
for f_path in glob.glob(f"{DIR_BASE}/clima_historico_distribuido/*europa*.json"):
    with open(f_path, "r", encoding="utf-8") as f:
        lista_europa.extend(json.load(f))
df_clima_europa = pd.DataFrame(lista_europa)

df_clima_global = pd.concat([df_clima_america, df_clima_sql, df_clima_europa], ignore_index=True)
print(f"Clima Global Consolidado: {df_clima_global.shape[0]} registros")



In [ ]:
# 1.6 Datos Simulados y IoT
print("Cargando datos simulados (Biometría, Tickets, Movilidad)...")
# Tickets
df_tickets = pd.concat([
    pd.read_csv(f"{DIR_BASE}/datos_simulados/tickets_parte1.csv"),
    pd.read_csv(f"{DIR_BASE}/datos_simulados/tickets_parte2.csv", header=None, names=["ID_Ticket", "Fecha_Compra", "ID_Evento", "Precio_USD", "Categoria"])
], ignore_index=True)
# Limpieza de tickets (Precios negativos)
df_tickets['Precio_USD'] = df_tickets['Precio_USD'].abs()

# Movilidad (Parquet)
df_movilidad = pd.read_parquet(f"{DIR_BASE}/datos_simulados/movilidad_urbana.parquet")

# Biometría JSON
lista_bio = []
for f_path in glob.glob(f"{DIR_BASE}/datos_simulados/biometria_chunk_*.json"):
    with open(f_path, "r", encoding="utf-8") as f:
        lista_bio.extend(json.load(f))
df_biometria = pd.DataFrame(lista_bio)
# Limpieza básica
df_biometria['BPM'] = pd.to_numeric(df_biometria['BPM'].replace("Error_Sensor", np.nan))
df_biometria = df_biometria[df_biometria['BPM'] < 300] # Quitar outliers masivos

print(f"Tickets: {df_tickets.shape[0]}, Movilidad: {df_movilidad.shape[0]}, Biometría: {df_biometria.shape[0]}")



In [ ]:
# 1.7 Redes Sociales y GDELT (Noticias Mundiales)
print("Cargando Tweets y GDELT...")
# GDELT
df_gdelt = pd.read_csv(f"{DIR_BASE}/googleBigQuery/bq-results-20260716-171541-1784222468943.csv")

# TWEETS
archivos_tweets = glob.glob(f"{DIR_BASE}/tweets/*.csv")
lista_tweets = []
for archivo in archivos_tweets:
    try:
        # Algunos archivos no tienen header en la partición 2, pero para simplificar usamos error_bad_lines=False
        df_t = pd.read_csv(archivo, on_bad_lines='skip', engine='python')
        # Estandarizar nombre de columna de texto
        if 'text' in df_t.columns:
            lista_tweets.append(df_t[['text']].dropna())
    except:
        pass

df_tweets = pd.concat(lista_tweets, ignore_index=True)

# Limpieza avanzada (NLP-prep)
def limpiar_texto(texto):
    texto = str(texto)
    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto, flags=re.MULTILINE) # urls
    texto = re.sub(r'\@\w+', '', texto) # menciones
    texto = re.sub(r'[^a-zA-Z\s]', '', texto) # dejar solo letras para NLP
    return texto.strip()

print("Limpiando texto de tweets (esto puede tomar un minuto)...")
# Tomamos una muestra aleatoria para que el notebook corra rápido, si es necesario, comenta el sample
df_tweets_sample = df_tweets.sample(n=50000, random_state=42).copy()
df_tweets_sample['text_clean'] = df_tweets_sample['text'].apply(limpiar_texto)

print(f"GDELT: {df_gdelt.shape[0]} registros")
print(f"Tweets limpios (Muestra): {df_tweets_sample.shape[0]}")



---
## Fase 2: Análisis de Sentimientos (NLP)
Vamos a calcular la polaridad de los tweets utilizando `TextBlob` y analizaremos el tono global de GDELT.



In [ ]:
# 2.1 Sentimiento en Twitter
def obtener_sentimiento(texto):
    try:
        return TextBlob(texto).sentiment.polarity
    except:
        return 0.0

print("Calculando sentimiento en Tweets...")
df_tweets_sample['Polaridad'] = df_tweets_sample['text_clean'].apply(obtener_sentimiento)

# Clasificación categórica
condiciones = [
    (df_tweets_sample['Polaridad'] > 0.1),
    (df_tweets_sample['Polaridad'] < -0.1)
]
opciones = ['Positivo', 'Negativo']
df_tweets_sample['Sentimiento'] = np.select(condiciones, opciones, default='Neutral')

display(df_tweets_sample[['text_clean', 'Polaridad', 'Sentimiento']].head())



---
## Fase 3: Métricas Gráficas y Visualización
A continuación presentaremos paneles visuales que cruzan múltiples dominios del proyecto.



In [ ]:
# 3.1 Distribución del Sentimiento en Twitter (Gráfico de Pastel)
plt.figure(figsize=(8, 6))
conteo_sentimientos = df_tweets_sample['Sentimiento'].value_counts()
plt.pie(conteo_sentimientos, labels=conteo_sentimientos.index, autopct='%1.1f%%', colors=['#cccccc', '#4CAF50', '#F44336'], startangle=140)
plt.title("Distribución de Sentimientos sobre las Olimpiadas en Twitter")
plt.axis('equal')
plt.show()



In [ ]:
# 3.2 Tono Global de Noticias GDELT por País
top_paises_noticias = df_gdelt['pais_actor_1'].value_counts().head(10).index
df_gdelt_top = df_gdelt[df_gdelt['pais_actor_1'].isin(top_paises_noticias)]

plt.figure(figsize=(14, 6))
sns.boxplot(x='pais_actor_1', y='sentimiento_promedio', data=df_gdelt_top, palette="Set2")
plt.title("Sentimiento Promedio de Noticias GDELT (Top 10 Países con más eventos)")
plt.xlabel("País Actor")
plt.ylabel("Tono / Sentimiento Promedio")
plt.axhline(0, color='red', linestyle='--')
plt.show()



In [ ]:
# 3.3 Relación Biometría IoT (BPM vs O2)
# Usando una muestra para la gráfica de dispersión
df_bio_sample = df_biometria.dropna().sample(n=10000, random_state=42)

plt.figure(figsize=(10, 6))
sns.scatterplot(x='BPM', y='Saturacion_O2', data=df_bio_sample, alpha=0.3, color='purple')
plt.title("Dispersión de Ritmo Cardíaco vs Saturación de O2 (IoT Simulado)")
plt.xlabel("Latidos por Minuto (BPM)")
plt.ylabel("Saturación de Oxígeno (%)")
plt.show()



In [ ]:
# 3.4 Clima Histórico (Variación de Temperaturas)
plt.figure(figsize=(12, 6))
sns.histplot(df_clima_global['temperature_2m'].dropna(), bins=50, kde=True, color='orange')
plt.title("Distribución Global de Temperaturas en Sedes y Ciudades Evaluadas")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Frecuencia")
plt.show()



In [ ]:
# 3.5 Top 10 Países Históricos en Medallas vs París 2024
top_hist = df_medals_hist['country_name'].value_counts().head(10)
top_paris = df_paris_medals['country_long'].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(x=top_hist.values, y=top_hist.index, ax=axes[0], palette="Blues_r")
axes[0].set_title("Top 10 Medallas Históricas (1896-2022)")
axes[0].set_xlabel("Total Medallas")

sns.barplot(x=top_paris.values, y=top_paris.index, ax=axes[1], palette="Reds_r")
axes[1].set_title("Top 10 Medallas en París 2024")
axes[1].set_xlabel("Total Medallas")

plt.tight_layout()
plt.show()



In [ ]:
# 3.6 Análisis Demográfico: Tendencia del PIB Global a lo largo del tiempo
# Promedio global de PIB per cápita por año
pib_trend = df_pib.groupby('anio')['pib_per_capita_usd'].mean().reset_index()
# Convertir año a numérico ignorando errores
pib_trend['anio'] = pd.to_numeric(pib_trend['anio'], errors='coerce')
pib_trend = pib_trend.dropna().sort_values('anio')

plt.figure(figsize=(12, 6))
sns.lineplot(x='anio', y='pib_per_capita_usd', data=pib_trend, marker='o', color='green')
plt.title("Crecimiento Promedio Global del PIB per Cápita (Datos Banco Mundial)")
plt.xlabel("Año")
plt.ylabel("PIB Per Cápita (USD)")
plt.grid(True)
plt.show()

print("¡Análisis End-to-End completado con éxito! 🎉")

